## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key


In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="Gemma2-9b-It")
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000226BDB09FC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000226BDB09E70>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Slim and I am a Chief AI Engineer")])

AIMessage(content="Hello Slim, it's nice to meet you!  \n\nAs a Chief AI Engineer, what kind of projects are you currently working on? \n\nI'm always interested in learning about the latest developments in AI.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 22, 'total_tokens': 70, 'completion_time': 0.087272727, 'prompt_time': 0.002143935, 'queue_time': 0.24368896499999998, 'total_time': 0.089416662}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--a85aed9d-86c9-455d-8c72-b05244e0c64c-0', usage_metadata={'input_tokens': 22, 'output_tokens': 48, 'total_tokens': 70})

In [4]:
model.invoke([HumanMessage(content="Hey What's my name and what do I do?")])

AIMessage(content="As an AI, I have no memory of past conversations and no access to personal information about you, including your name or occupation.\n\nIf you'd like to tell me your name and what you do, I'd be happy to know! 😄  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 21, 'total_tokens': 77, 'completion_time': 0.101818182, 'prompt_time': 0.002134385, 'queue_time': 0.243725655, 'total_time': 0.103952567}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--788a82f9-53f2-4f5c-bdaa-199e91a01fb6-0', usage_metadata={'input_tokens': 21, 'output_tokens': 56, 'total_tokens': 77})

In [8]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Slim and I am a Chief AI Engineer"),
        AIMessage(content="Hello Krish! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="You are Slim, and you're a Chief AI Engineer! \n\nIt was nice of you to remind me. 😊  \n\nIs there anything I can help you with related to your work today?  Perhaps you'd like to brainstorm ideas, explore some AI concepts, or even just chat about the latest advancements in the field?  \n\n\n\n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 97, 'total_tokens': 170, 'completion_time': 0.132727273, 'prompt_time': 0.004349282, 'queue_time': 0.24346543699999998, 'total_time': 0.137076555}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--d0418c2b-52d9-4d28-bb08-f04e18e96cc7-0', usage_metadata={'input_tokens': 97, 'output_tokens': 73, 'total_tokens': 170})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [ ]:
#!pip install langchain_community

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [6]:
config={"configurable":{"session_id":"chat1"}}

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Slim and I am a Chief AI Engineer")],
    config=config
)

In [8]:
response.content

"Hello Slim! It's nice to meet you.  \n\nThat's an impressive title! As a large language model, I'm always interested in learning more about the work of AI engineers. What kinds of projects are you working on these days?\n"

In [9]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
)

AIMessage(content='You told me your name is Slim!  😊  \n\nIs there something else I can help you with?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 91, 'total_tokens': 116, 'completion_time': 0.045454545, 'prompt_time': 0.004310681, 'queue_time': 0.24367430899999998, 'total_time': 0.049765226}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--9890684f-f78b-4130-afaa-f3c2da65f427-0', usage_metadata={'input_tokens': 91, 'output_tokens': 25, 'total_tokens': 116})

In [10]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"As an AI, I have no memory of past conversations and do not know your name. If you'd like to tell me, I'd be happy to use it! 😊\n"

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

"Hi John! It's nice to meet you. 👋  \n\nWhat can I do for you today?\n"

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John!  😊 \n\nIs there anything else I can help you with, John?\n'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [13]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [14]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Slim")]})

AIMessage(content="Hi Slim! \n\nIt's nice to meet you. \n\nWhat can I do for you today? I'm ready to answer your questions and help in any way I can.  😊 \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 31, 'total_tokens': 76, 'completion_time': 0.081818182, 'prompt_time': 0.002395155, 'queue_time': 0.243855084, 'total_time': 0.084213337}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--d53889e4-a253-4ecf-a7ea-879a0963673a-0', usage_metadata={'input_tokens': 31, 'output_tokens': 45, 'total_tokens': 76})

In [15]:
chain.invoke(["Hi My name is Slim"]) #Simple chat

AIMessage(content="Hi Slim, it's nice to meet you! \n\nWhat can I do for you today? I'm ready for any questions you have. 😊  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 31, 'total_tokens': 68, 'completion_time': 0.067272727, 'prompt_time': 0.002829815, 'queue_time': 0.42786500499999996, 'total_time': 0.070102542}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--18963ff5-b282-4263-8dcc-fa6ab74c729f-0', usage_metadata={'input_tokens': 31, 'output_tokens': 37, 'total_tokens': 68})

In [16]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [17]:
config = {"configurable":{"session_id":"chat3"}}

response = with_message_history.invoke(
    [HumanMessage(content= "Hi, my name is slim.")],
                                       config=config
)

In [18]:
response

AIMessage(content="Hello Slim! It's nice to meet you.\n\nI'm ready to answer your questions to the best of my ability.  What can I do for you? 😊 \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 33, 'total_tokens': 73, 'completion_time': 0.072727273, 'prompt_time': 0.003415214, 'queue_time': 0.24841397599999998, 'total_time': 0.076142487}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--c3feb9a4-6cfe-4640-bbdd-0d768552f55e-0', usage_metadata={'input_tokens': 33, 'output_tokens': 40, 'total_tokens': 73})

In [19]:
response = with_message_history.invoke(
    [HumanMessage(content="what's my name?")],
    config=config,
)

response.content

'Your name is Slim!  You told me at the beginning.  😄  \n'

### Add more complexity

In [20]:
prompt= ChatPromptTemplate.from_messages(
    [
        (
            "system", "Answer all questions to the best of your ability in {language}."),
    MessagesPlaceholder("messages")

    ]
)

In [21]:
chain = prompt|model

In [23]:
response = chain.invoke(
    {
        "messages": [HumanMessage(content="Hi, my name is slim")],
        "language": "Hindi"
    }
)
response.content

'नमस्ते, Slim! \n\nमुझे ख़ुशी है आपसे मिलने की।  \n\nआप मुझे क्या पूछना चाहते हैं?  \n\n'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [24]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [26]:
config = {"configurable": {"session_id": "chat4"}}

response = with_message_history.invoke(
    {"messages": [HumanMessage(content="Hi, my name is slim")], 
     "language": "Hindi"
     },
    config=config
)
response.content

'नमस्ते, स्लिम!  \n\nआपका नाम सुनकर बहुत अच्छा लगा। 😊 \n'

In [27]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

response.content

'आपका नाम स्लिम है। 😊  \n'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [49]:
from langchain_core.messages import SystemMessage,trim_messages, AIMessage
trimmer=trim_messages(
    max_tokens=45,
    strategy="last", # Defaulf = "last"
    token_counter=model,
    include_system=True, # Defaulf = False
    allow_partial=False, # Defaulf = False
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages) 

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [50]:
prompt= ChatPromptTemplate.from_messages(
    [
        (
            "system", "Answer all questions to the best of your ability in {language}."),
    MessagesPlaceholder("messages")

    ]
)

In [51]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"As an AI, I don't have access to your personal information, including your ice cream preferences.  What's your favorite flavor?  🍦😄\n"

In [52]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked "what\'s 2 + 2". 😊  \n\n\n\nLet me know if you\'d like to try another one!\n'

In [53]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config={"configurable":{"session_id":"chat5"}}

In [54]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I don't have access to past conversations or any personal information about you, including your name.\n\nIf you'd like to tell me your name, I'd love to know! 😊  \n\n"

In [55]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I have no memory of past conversations. That means I don't know what math problem you asked me. \n\nIf you'd like to ask me a math problem now, I'm happy to help! 😊  \n\n"